# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZiadMGamal/Search-Ranking-ML-FlyRank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## My Lane: Ranking Lifecycle

I'm choosing **ranking_lifecycle** as my provisional lane.

**Why:** The starter dataset (`content_refresh_anonymized.csv`, 30,000 pages across 32 clients) is built around exactly this lifecycle — each page has 90-day activity totals, 30-day trend windows, and a pre-computed `trend_direction` (new / flat / up / down / stable). Over half the sample (54.2%) is currently trending down, which tells me there's a real, sizable population of pages that would benefit from a systematic "review this first" queue instead of manual, ad-hoc checking. This is also the largest lane on the full warehouse (~205K rows), giving me the most room to find robust archetypes before narrowing down.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## The Question

**Search question:** Given a page's recent search performance and content signals, which pages are the highest-priority candidates for manual content review this week?

**Unit of analysis:** One page (identified by an anonymized page ID), observed over a rolling 90-day window of impressions, clicks, and position.

**Output:** A ranked list — a "review queue" — of pages, ordered by priority score, with the top signals that drove each page's score.

**Decision this informs:** Which pages an SEO analyst or content team should look at first, out of potentially thousands of live pages, given limited weekly review capacity.

**Action someone takes:** A content strategist pulls the top N pages from the queue and manually reviews them — deciding whether to refresh content, fix technical issues, or deprioritize the page entirely.

**Cost of a wrong recommendation:**
- **False positive** (flagged as urgent, but it's actually fine): wastes analyst time — a low-to-moderate cost, but it adds up across many pages.
- **False negative** (a genuinely declining or important page is missed): the page keeps losing rankings and traffic silently until someone notices manually — a higher, compounding cost, since lost organic traffic doesn't recover for free.

**Why data/ML helps at all:** No human can manually track lifecycle patterns across thousands of pages every week. A model that scores and ranks pages turns an impossible-to-scale manual audit into a short, prioritized list. This is not about predicting Google's algorithm — it's about surfacing signal from data we already have.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
!git clone https://github.com/ZiadMGamal/Search-Ranking-ML-FlyRank.git repo
%cd repo

Cloning into 'repo'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 122 (delta 34), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 1.87 MiB | 13.95 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/repo/repo


In [5]:
import pandas as pd

df=pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:",df.shape)

n_pages=df.shape[0]
n_clients=df["client_id"].nunique()
print(f"Total pages: {n_pages} across {n_clients} pseudonymized clients")

decline_share= (df["trend_direction"] == "down").mean()
print(f"Share of pages currently trending down: {decline_share:.1%}")

top10_cutoff=int(len(df)*0.10)
top10_share =(
    df.sort_values("impressions_90d",ascending=False)
    .head(top10_cutoff)["impressions_90d"]
    .sum()
    / df["impressions_90d"].sum()
)
print(f"Share of total impressions held by the top 10% of pages: {top10_share:.1%}")

Shape: (30000, 44)
Total pages: 30000 across 32 pseudonymized clients
Share of pages currently trending down: 54.2%
Share of total impressions held by the top 10% of pages: 70.2%


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## Careful Words

**What I can claim:**
- This is an observed, historical pattern in a 30,000-row anonymized teaching slice — not a prediction of Google's ranking algorithm.
- ~54% of pages in this sample show a "down" trend, and the top 10% of pages by impressions account for ~70% of total search visibility — a real, measurable concentration worth prioritizing around.
- The priority score I'll eventually build is decision-support for a human reviewer, not an autonomous action.

**What I can't claim (and why — specific to this dataset):**
- I cannot use `trend_direction` or `trend_pct` as model features later — they ARE the label (`is_declining_label`), so using them would be leakage, not prediction.
- I cannot treat missing keyword-context fields (`search_volume`, `competition`, `cpc`) as zero — they're blank because ~2,468 rows have no keyword data at all, tied to content type (e.g. feedly articles). A blind `fillna(0)` would silently encode content type into my features.
- I cannot read `position_tier` medians without checking the volume floor — for example, the "top_3" tier's median volume is only ~53 impressions/90d in this slice, where a single click swings CTR by ~1.9 percentage points. A tier label alone can be statistically noisy at low volume.
- I cannot claim to have "predicted Google's algorithm" — I only have GSC/GA4 outcome signals (impressions, clicks, position, engagement), not the algorithm's inputs.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.